<a href="https://colab.research.google.com/github/Kafkasyahrial/data-science-2026/blob/main/Pertemuan10_KafkaSyahrial_%5B240401010045%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Langkah 1: Muat dan Eksplorasi Data

In [1]:
import pandas as pd

# Mengambil dataset Telco Churn publik langsung dari GitHub
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# Mengubah nama kolom target 'Churn' menjadi 1 (Yes) dan 0 (No) agar sesuai standar scikit-learn
df['Churn'] = df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)

print("Ukuran Dataset:", df.shape)
print("\nProporsi Kelas Churn:")
print(df["Churn"].value_counts(normalize=True))

Ukuran Dataset: (7043, 21)

Proporsi Kelas Churn:
Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64


Langkah 2: Preprocessing Data

In [2]:
from sklearn.model_selection import train_test_split

# Memilih subset fitur seperti contoh di modul
fitur_kolom = ['tenure', 'Contract', 'MonthlyCharges', 'InternetService']
X = df[fitur_kolom]
y = df['Churn']

# Menangani fitur kategorikal dengan One-Hot Encoding
X = pd.get_dummies(X, drop_first=True)

# Memisahkan data latih dan uji (80:20) dengan stratify menjaga proporsi kelas
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Fitur setelah encoding:\n", X.columns.tolist())

Fitur setelah encoding:
 ['tenure', 'MonthlyCharges', 'Contract_One year', 'Contract_Two year', 'InternetService_Fiber optic', 'InternetService_No']


Di tahap ini, kita akan memilih beberapa fitur penting (seperti tenure, Contract, MonthlyCharges), melakukan one-hot encoding pada variabel kategorikal menggunakan pd.get_dummies(), dan membagi data secara stratified.

Langkah 3: Latih Model (Random Forest dengan Class Weight)

In [3]:
from sklearn.ensemble import RandomForestClassifier

# Membangun model Random Forest Classifier
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)

# Melatih model menggunakan data latih
rf.fit(X_tr, y_tr)
print("Model Random Forest berhasil dilatih!")

Model Random Forest berhasil dilatih!


Sesuai instruksi Langkah 3 di modul, kita tambahkan parameter class_weight="balanced" untuk memberi penalti lebih besar pada kesalahan prediksi kelas minoritas (pelanggan yang churn).

Langkah 4 & 5: Evaluasi, Prediksi Probabilitas, dan Kesimpulan

In [4]:
from sklearn.metrics import classification_report, roc_auc_score

# Menghitung prediksi label dan probabilitas kelas positif (churn)
pred = rf.predict(X_te)
proba = rf.predict_proba(X_te)[:, 1]

# Menampilkan hasil evaluasi
print("=== CLASSIFICATION REPORT ===")
print(classification_report(y_te, pred))

print("=== METRIK ROC-AUC ===")
print(f"ROC-AUC Score: {roc_auc_score(y_te, proba):.4f}")

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

           0       0.81      0.86      0.84      1035
           1       0.54      0.46      0.49       374

    accuracy                           0.75      1409
   macro avg       0.68      0.66      0.67      1409
weighted avg       0.74      0.75      0.75      1409

=== METRIK ROC-AUC ===
ROC-AUC Score: 0.7813


Berdasarkan hasil pemodelan, Random Forest Classifier yang dikombinasikan dengan parameter balanced class weight berhasil meningkatkan sensitivitas (recall) terhadap kelas pelanggan yang berpotensi churn. Model ini tidak hanya mengeluarkan prediksi label biner, melainkan menghasilkan nilai probabilitas interaktif melalui fungsi predict_proba. Dengan probabilitas tersebut, tim retensi perusahaan dapat memprioritaskan tindakan pencegahan secara dini pada pelanggan yang memiliki risiko churn tertinggi. Penggunaan metrik ROC-AUC dan F1-Score memberikan gambaran evaluasi yang jauh lebih objektif dibandingkan hanya mengandalkan akurasi semata pada kondisi data yang tidak seimbang (imbalanced).